# 01d — Cd44 source, and the metabolic state of Spp1+ tumour vs Spp1+ macrophages

All in **single-cell** (`annotation_v4.h5ad`) — dissociated, so no segmentation spillover.
Three questions the spatial data cannot answer cleanly:

1. **Who produces Cd44** — the receptor side of the axis 01c showed is tumour-driven on the
   ligand side. Together they say *tumour Spp1 -> whose CD44?*
2. **Metabolic state of Spp1-high vs Spp1-low cancer cells** — is the tumour's Spp1 tied to a
   lipid/hypoxia program, or incidental?
3. **Do Spp1+ cancer cells and SPP1-TAM macrophages share a metabolic state?** — the real
   question. If the same program is induced in the Spp1-high fraction of *both* compartments,
   that is convergence onto a shared metabolic niche, which is the metabolism story's core.

### The one methodological trap, stated up front
The 01c `SPP1_TAM_sig` was **selected to be macrophage-specific** (>2x vs other types). Scoring
it on tumour cells is rigged to return ~0 and would "disprove" convergence by construction. So
convergence here is tested the honest way: **Spp1-high-vs-low DE run independently in each
compartment, then the fold-change vectors correlated.** Same genes moving the same way = a
shared program. Metabolic pathway scores are the interpretable companion, not the test.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

import scanpy as sc
import anndata as ad

from scipy.stats import mannwhitneyu, spearmanr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)

## 1. Config + load (same reference as 01c)

In [ ]:
SC_REF = Path("/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/"
              "annotating_references/annotation_v4.h5ad")
PATHWAY_DIR = Path("/coh_labs/yunroseli/Jona/CAR-T/data/references/pathways")
OUT_DIR = Path("/coh_labs/yunroseli/Jona/CAR-T/results/spp1_analysis")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

GMT_FILES = {
    "hallmark": PATHWAY_DIR / "mh.all.v2026.1.Mm.symbols.gmt",
    "reactome": PATHWAY_DIR / "m2.cp.reactome.v2026.1.Mm.symbols.gmt",
    "go_bp":    PATHWAY_DIR / "m5.go.bp.v2026.1.Mm.symbols.gmt",
    "curated":  PATHWAY_DIR / "m2.cp.v2026.1.Mm.symbols.gmt",
}

TUMOUR_LABEL = "Cancer_cell"
MAC_LABELS   = ["M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"]
HI_Q, LO_Q   = 0.75, 0.25
MIN_CELLS_DE = 100

assert SC_REF.exists(), f"reference not found: {SC_REF}"
scref = sc.read_h5ad(SC_REF)
print(scref)

# level auto-detect (same logic as 01c) — pick the lvl column with M1/M2/Cancer_cell
lvl_cols = [c for c in scref.obs.columns if c.startswith("cell_type_lvl")]
want = set(MAC_LABELS) | {TUMOUR_LABEL}
CT_COL = max(lvl_cols, key=lambda c: len(set(scref.obs[c].astype(str)) & want))
sc_ct = scref.obs[CT_COL].astype(str)
print(f"\nusing {CT_COL} | genes: {scref.n_vars:,}"
      + ("" if scref.n_vars > 20000 else "  <- gene-filtered; some metabolic genes may be absent"))
for lab in [TUMOUR_LABEL] + MAC_LABELS:
    print(f"  {lab}: {int((sc_ct == lab).sum()):,}")

# normalize a working copy if raw
X = scref.X
head = X[:200].toarray() if sp.issparse(X) else X[:200]
if bool(np.allclose(head, np.round(head))) and float(X.max()) > 50:
    scref.layers["counts"] = scref.X.copy()
    sc.pp.normalize_total(scref, target_sum=1e4); sc.pp.log1p(scref)
    print("\nnormalized raw counts -> CP10K + log1p")
else:
    print("\n.X already log-normalized")

def expr_of(a, gene):
    v = a[:, gene].X
    return np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v).ravel()

## 2. Q1 — who produces Cd44?

Same per-cell + projected-to-tissue logic as 01c's Spp1 gate. Cd44 is a receptor and is
*legitimately* broad (activated lymphoid cells express it), so unlike Spp1 the point is not
"who is specific" but **which cell types form the dominant CD44 pool the tumour Spp1 could
signal to**. The ambient floor (~0.4 in 01c) still applies — read against it.

In [ ]:
SPATIAL_N = {
    "Cancer_cell": 363019, "Erythrocyte": 210019, "N2_like_Neu": 193240, "N1_like_Neu": 134656,
    "M2_like_Mac": 60720, "Classical_Mono": 58279, "CD8_T": 45824, "B": 40422,
    "Fibroblast": 24867, "NK": 22924, "M1_like_Mac": 19170, "Endothelial": 17702,
    "CD4_T": 10085, "cDC": 8471, "NKT": 7110, "Treg": 5155, "pDC": 4558,
    "Nonclassical_Mono": 4002, "Intermediate_Mac": 1675,
}
assert "Cd44" in scref.var_names, "Cd44 not in var_names"
scref.obs["Cd44_expr"] = expr_of(scref, "Cd44")
scref.obs["Spp1_expr"] = expr_of(scref, "Spp1")

cd44 = (pd.DataFrame({"ct": sc_ct.values, "Cd44": scref.obs["Cd44_expr"].values})
        .groupby("ct")
        .agg(n_sc=("Cd44", "size"), mean_Cd44=("Cd44", "mean"),
             pct_detected=("Cd44", lambda s: 100 * (s > 0).mean())))
cd44["n_spatial"] = cd44.index.map(SPATIAL_N)
cd44["proj_Cd44"] = cd44["mean_Cd44"] * cd44["n_spatial"]
cd44["pct_of_tissue_Cd44"] = 100 * cd44["proj_Cd44"] / cd44["proj_Cd44"].sum()
cd44["reliable"] = cd44["n_sc"] >= 200
cd44 = cd44.sort_values("pct_of_tissue_Cd44", ascending=False)
cd44.to_csv(OUT_DIR / "sc_cd44_by_celltype.csv")

print("=== who carries Cd44 (per-cell x real spatial abundance) ===")
print(cd44.round(3).to_string())

tum_cd44 = cd44.loc[TUMOUR_LABEL, "pct_of_tissue_Cd44"] if TUMOUR_LABEL in cd44.index else np.nan
mac_cd44 = cd44.loc[[m for m in MAC_LABELS if m in cd44.index], "pct_of_tissue_Cd44"].sum()
print(f"""
tissue Cd44 pool: Cancer_cell {tum_cd44:.1f}%  |  macrophages {mac_cd44:.1f}%
  tumour high on BOTH Spp1 and Cd44 -> autocrine/juxtacrine tumour Spp1-CD44 loop; the deck's
      "interaction-high clusters" are largely tumour talking to itself.
  Cd44 spread across immune/stroma -> tumour Spp1 signals OUTWARD to a CD44+ compartment; then
      WHICH compartment (macs? T cells?) is the biology worth chasing.
""")

In [ ]:
# Figure: Spp1 source (from 01c) vs Cd44 source, side by side.
spp1 = (pd.DataFrame({"ct": sc_ct.values, "Spp1": scref.obs["Spp1_expr"].values})
        .groupby("ct")["Spp1"].mean() * pd.Series(SPATIAL_N))
spp1 = 100 * spp1 / spp1.sum()
order = cd44.sort_values("pct_of_tissue_Cd44", ascending=False).index[:14]
y = np.arange(len(order))
colr = ["#2ca02c" if i == TUMOUR_LABEL else ("#d62728" if i in MAC_LABELS else "#7f7f7f") for i in order]
fig, axes = plt.subplots(1, 2, figsize=(11, 0.42 * len(order) + 2), sharey=True)
axes[0].barh(y, spp1.reindex(order).values, color=colr); axes[0].set_title("% of tissue Spp1 (ligand)", fontsize=10)
axes[1].barh(y, cd44.reindex(order)["pct_of_tissue_Cd44"].values, color=colr); axes[1].set_title("% of tissue Cd44 (receptor)", fontsize=10)
axes[0].set_yticks(y); axes[0].set_yticklabels(order, fontsize=8); axes[0].invert_yaxis()
fig.suptitle("The Spp1-CD44 axis: ligand source vs receptor source\n(green=tumour, red=macrophage)", y=1.04)
fig.tight_layout(); fig.savefig(FIG_DIR / "sc_spp1_cd44_sources.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)

## 3. Metabolic pathway panel (scored on all cells)

The metabolism-angle pathways, verified present in the GMTs in earlier passes. Scored once on
the whole object; §4 and §5 read them back per compartment.

In [ ]:
METAB_PANEL = {
    "Hypoxia_Hallmark": "HALLMARK_HYPOXIA",
    "Glycolysis_Hallmark": "HALLMARK_GLYCOLYSIS",
    "OxPhos_Hallmark": "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
    "FattyAcid_Metab_Hallmark": "HALLMARK_FATTY_ACID_METABOLISM",
    "Cholesterol_Homeo_Hallmark": "HALLMARK_CHOLESTEROL_HOMEOSTASIS",
    "Adipogenesis_Hallmark": "HALLMARK_ADIPOGENESIS",
    "mTORC1_Hallmark": "HALLMARK_MTORC1_SIGNALING",
    "FAO_GO": "GOBP_FATTY_ACID_BETA_OXIDATION",
    "LipidDroplet_GO": "GOBP_LIPID_DROPLET_ORGANIZATION",
    "oxLDL_Response_GO": "GOBP_CELLULAR_RESPONSE_TO_OXIDISED_LOW_DENSITY_LIPOPROTEIN_PARTICLE_STIMULUS",
    "Cholesterol_Efflux_GO": "GOBP_CHOLESTEROL_EFFLUX",
    "Lactate_Metab_GO": "GOBP_LACTATE_METABOLIC_PROCESS",
    "Adenosine_Metab_GO": "GOBP_ADENOSINE_METABOLIC_PROCESS",
    "Glutamine_Metab_GO": "GOBP_GLUTAMINE_METABOLIC_PROCESS",
}

def read_gmt(p):
    out = {}
    for line in open(p):
        x = line.rstrip("\n").split("\t")
        if len(x) >= 3: out[x[0]] = x[2:]
    return out

gene_sets = {}
for _, p in GMT_FILES.items():
    gene_sets.update(read_gmt(p))

available = set(scref.var_names.astype(str))
metab_cols = []
for short, gs in METAB_PANEL.items():
    matched = [g for g in gene_sets.get(gs, []) if g in available]
    if len(matched) < 5:
        print(f"[skip] {short}: {len(matched)} genes"); continue
    sc.tl.score_genes(scref, gene_list=matched, score_name=short, use_raw=False, random_state=0)
    metab_cols.append(short)
print(f"{len(metab_cols)}/{len(METAB_PANEL)} metabolic pathways scored")

## 4. Q2 + Q3 setup — split each compartment by Spp1, score everything

Within tumour and within macrophages separately, split by Spp1 quantile (same as 01c). Gives
four groups: {tumour, mac} x {Spp1_high, Spp1_low}.

In [ ]:
def spp1_split(mask):
    e = scref.obs.loc[mask, "Spp1_expr"]
    hi, lo = np.quantile(e, HI_Q), np.quantile(e, LO_Q)
    grp = pd.Series("mid", index=e.index)
    grp[e >= hi] = "Spp1_high"; grp[e <= lo] = "Spp1_low"
    return grp

scref.obs["compartment"] = "other"
scref.obs.loc[sc_ct == TUMOUR_LABEL, "compartment"] = "tumour"
scref.obs.loc[sc_ct.isin(MAC_LABELS), "compartment"] = "mac"
scref.obs["spp1_grp"] = "NA"
for comp, mask in [("tumour", (sc_ct == TUMOUR_LABEL).values),
                   ("mac", sc_ct.isin(MAC_LABELS).values)]:
    scref.obs.loc[mask, "spp1_grp"] = spp1_split(mask).values

scref.obs["comp_grp"] = scref.obs["compartment"] + "_" + scref.obs["spp1_grp"]
print(scref.obs.loc[scref.obs["compartment"] != "other", "comp_grp"].value_counts().to_string())

# Metabolic scores across the four groups.
grp_means = (scref.obs[scref.obs["comp_grp"].str.contains("Spp1")]
             .groupby("comp_grp")[metab_cols].mean().T)
grp_means = grp_means[[c for c in ["tumour_Spp1_high", "tumour_Spp1_low",
                                   "mac_Spp1_high", "mac_Spp1_low"] if c in grp_means.columns]]
grp_means.to_csv(OUT_DIR / "sc_metabolic_scores_by_group.csv")

print("\n=== metabolic pathway scores: mean per group ===")
print(grp_means.round(3).to_string())

In [ ]:
# Heatmap of the 2x2. The comparison that answers Q2 is tumour_high vs tumour_low; Q3 is
# whether the tumour_high and mac_high COLUMNS look alike.
data = grp_means.values.astype(float)
vmax = np.nanmax(np.abs(data)) or 1.0
fig, ax = plt.subplots(figsize=(1.5 * data.shape[1] + 3, 0.42 * data.shape[0] + 2))
im = ax.imshow(data, cmap="RdBu_r", norm=TwoSlopeNorm(0, -vmax, vmax), aspect="auto")
ax.set_xticks(range(data.shape[1])); ax.set_xticklabels(grp_means.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(data.shape[0])); ax.set_yticklabels(grp_means.index, fontsize=8)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f"{data[i,j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Metabolic state by compartment x Spp1 status", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
fig.tight_layout(); fig.savefig(FIG_DIR / "sc_metabolic_2x2.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)

## 5. Q2 — DE: Spp1-high vs Spp1-low cancer cells

Is the tumour's Spp1 tied to a metabolic program, or incidental?

In [ ]:
def de_high_vs_low(mask, tag):
    sub = scref[mask & scref.obs["spp1_grp"].isin(["Spp1_high", "Spp1_low"]).values].copy()
    n_hi = int((sub.obs["spp1_grp"] == "Spp1_high").sum())
    n_lo = int((sub.obs["spp1_grp"] == "Spp1_low").sum())
    if n_hi < MIN_CELLS_DE or n_lo < MIN_CELLS_DE:
        print(f"[{tag}] too few cells (hi={n_hi}, lo={n_lo})"); return None
    sc.pp.filter_genes(sub, min_cells=10)
    sc.tl.rank_genes_groups(sub, "spp1_grp", groups=["Spp1_high"], reference="Spp1_low",
                            method="wilcoxon", pts=True)
    r = sub.uns["rank_genes_groups"]
    df = pd.DataFrame({"gene": [x[0] for x in r["names"]],
                       "log2FC": [x[0] for x in r["logfoldchanges"]],
                       "score": [x[0] for x in r["scores"]],
                       "pval_adj": [x[0] for x in r["pvals_adj"]]})
    print(f"[{tag}] DE done: {n_hi} high vs {n_lo} low")
    return df

de_tum = de_high_vs_low((sc_ct == TUMOUR_LABEL).values, "tumour")
de_mac = de_high_vs_low(sc_ct.isin(MAC_LABELS).values, "mac")
de_tum.to_csv(OUT_DIR / "sc_de_spp1_tumour.csv", index=False)
de_mac.to_csv(OUT_DIR / "sc_de_spp1_mac.csv", index=False)
assert de_tum is not None and de_mac is not None, \
    "DE skipped: a compartment had <MIN_CELLS_DE Spp1-high/low cells; convergence (cell below) needs both."

print("\n=== Spp1-high vs Spp1-low CANCER cells — top 25 UP ===")
print(de_tum.nlargest(25, "score")[["gene", "log2FC", "pval_adj"]].round(3).to_string(index=False))

# Do the metabolic programs move in Spp1-high tumour?
CHECK = {"lipid/TAM": ["Trem2","Apoe","Fabp5","Cd63","Lgals3","Lpl","Plin2","Cd36"],
         "cholesterol": ["Hmgcr","Ldlr","Scd1","Srebf2","Fdps"],
         "hypoxia/glyco": ["Hif1a","Vegfa","Ldha","Slc2a1","Pgk1"]}
idx = de_tum.set_index("gene")
print("\n--- metabolic markers in Spp1-high tumour ---")
for name, genes in CHECK.items():
    pres = [g for g in genes if g in idx.index]
    if pres:
        s = idx.loc[pres, ["log2FC", "pval_adj"]].sort_values("log2FC", ascending=False)
        print(f"\n  {name}: " + ", ".join(f"{g}({s.loc[g,'log2FC']:+.2f})" for g in s.index))

## 6. Q3 — DO THEY CONVERGE? (the honest test)

Correlate the Spp1-high-vs-low **fold-change vectors** of tumour and macrophages over their
shared genes. Positive correlation = the *same genes* move the *same way* when Spp1 is high in
either compartment: a shared metabolic program, not just shared Spp1.

Deliberately **not** using the mac-specific 01c signature — that would be rigged to say "no".

In [ ]:
m = de_tum.merge(de_mac, on="gene", suffixes=("_tum", "_mac"))
# focus on genes significant in at least one compartment
sig = m[(m["pval_adj_tum"] < 0.05) | (m["pval_adj_mac"] < 0.05)].copy()
rho, p = spearmanr(sig["log2FC_tum"], sig["log2FC_mac"])
r_pear = np.corrcoef(sig["log2FC_tum"], sig["log2FC_mac"])[0, 1]

# shared UP genes (the concrete convergent program)
up_tum = set(de_tum[(de_tum["log2FC"] > 0.5) & (de_tum["pval_adj"] < 0.05)]["gene"])
up_mac = set(de_mac[(de_mac["log2FC"] > 0.5) & (de_mac["pval_adj"] < 0.05)]["gene"])
shared = up_tum & up_mac
jacc = len(shared) / len(up_tum | up_mac) if (up_tum | up_mac) else np.nan

print(f"""=== convergence of the Spp1-high program: tumour vs macrophage ===
  genes compared (sig in >=1) : {len(sig):,}
  Spearman r (log2FC vectors) : {rho:+.3f}   (p={p:.1e})
  Pearson  r                  : {r_pear:+.3f}
  UP genes: tumour={len(up_tum)}  mac={len(up_mac)}  shared={len(shared)}  Jaccard={jacc:.3f}

  r strongly + (say >0.4) -> convergence: Spp1-high tumour and Spp1-high macs turn on the same
      program. The shared genes below ARE the shared metabolic state - the metabolism story.
  r ~ 0 -> no shared program; Spp1-high means different things in each compartment. Then the
      macrophage metabolic state stands on its own (01c §4) and is not tumour-mirrored.
""")
print("shared UP genes (Spp1-high in BOTH compartments):")
print(sorted(shared))

# annotate which shared genes are metabolic
metab_genes = set()
for gs in METAB_PANEL.values():
    metab_genes |= set(gene_sets.get(gs, []))
shared_metab = sorted(shared & metab_genes)
print(f"\nof those, in the metabolic panel ({len(shared_metab)}): {shared_metab}")

In [ ]:
# Scatter of the two fold-change vectors; shared metabolic genes labelled.
fig, ax = plt.subplots(figsize=(6.2, 6))
ax.scatter(sig["log2FC_tum"], sig["log2FC_mac"], s=6, alpha=0.25, color="#7f7f7f", linewidths=0)
lab = sig[sig["gene"].isin(shared_metab)]
ax.scatter(lab["log2FC_tum"], lab["log2FC_mac"], s=40, color="#d62728", zorder=3)
for _, r in lab.iterrows():
    ax.annotate(r["gene"], (r["log2FC_tum"], r["log2FC_mac"]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")
lim = np.nanpercentile(np.abs(np.r_[sig["log2FC_tum"], sig["log2FC_mac"]]), 99)
ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.8, alpha=0.5)
ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("log2FC  Spp1-high vs low  (TUMOUR)"); ax.set_ylabel("log2FC  Spp1-high vs low  (MAC)")
ax.set_title(f"Convergent Spp1-high program\nSpearman r = {rho:+.2f}", fontsize=11)
fig.tight_layout(); fig.savefig(FIG_DIR / "sc_convergence_scatter.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close(fig)

## 7. Summary

In [ ]:
print(f"""
==================== SUMMARY — report these back ====================
Q1  Cd44 source     : Cancer_cell {tum_cd44:.1f}% vs macrophages {mac_cd44:.1f}% of tissue Cd44 pool
                      (with 01c: tumour makes 53% of Spp1 too -> autocrine loop vs outward signalling)
Q2  Spp1-high tumour: {len(up_tum)} genes UP; metabolic markers listed in §5
Q3  CONVERGENCE     : Spearman r = {rho:+.3f} between tumour and mac Spp1-high fold-changes
                      {len(shared)} shared UP genes, {len(shared_metab)} of them metabolic

INTERPRETATION
 - r high  -> tumour and TAM converge on ONE Spp1-high metabolic program. Strongest possible
              framing for the metabolism angle: a shared niche state, not two coincidences.
 - r low   -> they do NOT share a program; the TAM metabolic state (01c §4) is its own thing.
              Still fundable, just not a tumour-mirrored convergence story.
 - watch Q1: if tumour carries most Cd44 too, the "SPP1-CD44 interaction-high" signal is largely
   tumour autocrine - which sharpens (not weakens) the reframe from notebook 01c §3b.

NOTE: single-cell is dissociated, so a shared program here is transcriptional convergence, not
proof of spatial co-location. Notebooks 02/03 test whether it plays out in tissue space.
=====================================================================
""")